AI Agent with RAG implementation

In [49]:
import os
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY= os.getenv("GOOGLE_API_KEY")

In [50]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs = []
data = Path("text_docs/")
for n in data.glob('*.pdf'):
    try:
        loader = PyMuPDFLoader(str(n))
        docs.extend(loader.load())
        print(f'file loaded sucessifuly: {n.name}')
    except Exception as e:
        print(f'failed to load the file {n.name}: {e}')

print(f'{len(docs)} docs loaded.')

file loaded sucessifuly: Política de Reembolsos (Viagens e Despesas).pdf
file loaded sucessifuly: Política de Uso de E-mail e Segurança da Informação.pdf
file loaded sucessifuly: Políticas de Home Office.pdf
3 docs loaded.


In [51]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)

In [52]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

embeddings = GoogleGenerativeAIEmbeddings(
    model = "models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [53]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_type= 'similarity_score_threshold',
                                     search_kwargs = {'score_threshold':0.3,'k':4})

In [54]:
llm_rag = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    api_key=GOOGLE_API_KEY
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [56]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

prompt_rag = ChatPromptTemplate.from_messages([
    ("system", "You are an expert HR and IT assistant for the company 'Torres Devs'. "
               "Your main task is to answer employee questions based ONLY on the context provided. "
               "Be polite and professional in your responses. "
               "If the information to answer the question is not in the provided context, "
               "clearly state that you cannot find the answer in the company's documents."),
    
    ("user", "Based on our company's policies, please answer the following question:\n\nQuestion: {input}\n\nContext:\n{context}")
])

document_chain = create_stuff_documents_chain(llm_rag, prompt_rag)

In [57]:
from typing import Dict

def askQuestionRag(question:str) -> Dict:
    related_docs = retriever.invoke(question)

    if not related_docs:
        return {"answer": "cannot find the answer in the company's documents",
                "citations":[],
                "context_found": False}
    
    answer = document_chain.invoke({'input': question,
                                   'context': related_docs})
    txt = (answer or "").strip()

    if txt.rstrip(".!?") == "cannot find the answer in the company's documents":
        return {"answer": "cannot find the answer in the company's documents",
                "citations":[],
                "context_found": False}
    
    return {"answer": txt,
                "citations":related_docs,
                "context_found": True}

In [ ]:
from pprint import pprint

tests = ["Posso reembolsar a internet?",
          "Me acidentei gravemente e estou enviando esse atestado",
          "Quero pedir demissão. Como faço?",
          "quantas cestas do logo o Stephen Curry fez na carreira?"]

for qst in tests:
    answer = askQuestionRag(qst)
    print(f'Question: {qst}')
    print(f'Answer: {answer['answer']}')
    if answer['context_found']:
        pprint(f'Citations: {answer['citations']}')
        print()